In [ ]:
#| default_exp deploy

In [ ]:
#| hide
import ast
from fastcore.test import *
from fastcore.all import Path, first
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Getting a web app onto a VM, as a pipeline over the project's own `deploy.py`.

`Deploy` runs the steps. `setup_template` and `deploy_template` write the two scripts those steps run, in the shape lego uses: a one-shot `setup.py` that registers the schema and pushes the secrets, and a `deploy.py` that CI runs on every push to main.

Nothing in this module reaches a cloud provider. The templates are text and the plan is commands. The packages that talk to Hetzner, Cloudflare and Docker are imported by the generated scripts, when those scripts run.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import os

In [ ]:
#| export
from pullup.env import EnvStore

In [ ]:
#| export
from pullup.env import env_value

In [ ]:
#| export
from pullup.pipeline import Pipeline, PipelineError

In [ ]:
#| export
from pullup.project import Step

In [ ]:
#| export
DEPLOY_FILE = 'deploy.json'

In [ ]:
#| export
_GENERATOR = os.environ.get('PULLUP_GENERATOR') or 'pullup'

def use_generator(name):
    "Name what the generated `setup.py` and `deploy.py` say wrote them: `use_generator('Leela')`."
    global _GENERATOR
    _GENERATOR = str(name)
    return _GENERATOR

def generator(): return _GENERATOR

In [ ]:
#| export
DeployError = PipelineError

`DeployError` is `PipelineError` under a second name. Catching either catches both.

`generator()` names the tool in the header of both generated files. It is written into somebody's repository, so it says what wrote it, and `use_generator` is how a host says its own name.


In [ ]:
#| export
DEPLOY_KEYS = {
    'MODE': 'prod', 'PORT': '5001', 'DOMAIN': None, 'SERVER_NAME': None, 'SERVER_USER': 'deploy',
    'HCLOUD_TOKEN': None, 'CLOUDFLARE_API_TOKEN': None, 'CF_TUNNEL_TOKEN': None,
    'RSYNC_FORCE': 'false',
}

`DEPLOY_KEYS` is the schema, and the only place a deploy's keys are declared. `None` means the value is a GitHub secret. A string means a GitHub variable with that default. `scaffold` merges the schema into the project's gheasy config, the generated workflow renders it into its `env:` block, and `.env` is the schema materialised.

A key with a default is never missing. `target` reports a key as missing only where the schema says `None` and the store has nothing.

In [ ]:
{k: ('secret' if v is None else f'variable, default {v!r}') for k, v in DEPLOY_KEYS.items()}

{'MODE': "variable, default 'prod'",
 'PORT': "variable, default '5001'",
 'DOMAIN': 'secret',
 'SERVER_NAME': 'secret',
 'SERVER_USER': "variable, default 'deploy'",
 'HCLOUD_TOKEN': 'secret',
 'CLOUDFLARE_API_TOKEN': 'secret',
 'CF_TUNNEL_TOKEN': 'secret',
 'RSYNC_FORCE': "variable, default 'false'"}

In [ ]:
#| export
DEPLOY_STEPS = [
    Step('env', 'write .env', 'python deploy.py env',
        'Materialise the schema into .env, which the container reads and the deploy uploads.'),
    Step('compose', 'build the stack', 'python deploy.py compose',
        'dockeasy writes the Dockerfile; vpseasy writes docker-compose.yml and the Caddyfile.'),
    Step('deploy', 'tunnel and server', 'python deploy.py deploy',
        'cfeasy opens the Cloudflare tunnel, vpseasy provisions the VPS if needed and rsyncs.',
        ['HCLOUD_TOKEN', 'CLOUDFLARE_API_TOKEN']),
]

Each step is one subcommand of the project's own `deploy.py`. The plan is commands rather than function calls, so a step reads the same way in a terminal, in a panel and in a workflow log. A `deploy.py` someone has edited is the deploy that runs.

The first two steps declare no keys. Writing `.env` and building the stack are local, and a project can get that far before it has a token for anything.

In [ ]:
[(s.id, s.cmd, s.needs) for s in DEPLOY_STEPS]

[('env', 'python deploy.py env', []),
 ('compose', 'python deploy.py compose', []),
 ('deploy',
  'python deploy.py deploy',
  ['HCLOUD_TOKEN', 'CLOUDFLARE_API_TOKEN'])]

In [ ]:
#| hide
needed = {k for s in DEPLOY_STEPS for k in s.needs}
assert needed <= set(DEPLOY_KEYS), f'a step needs keys the schema never declares: {needed - set(DEPLOY_KEYS)}'

In [ ]:
#| export
def setup_template(app, domain, subdomain='', port=5001, lfs=True):
    "lego's `setup.py`: the one-shot half of the pattern, with `ENV_KEYS` as its single source."
    host = f'{subdomain}.{domain}' if subdomain else domain
    patterns = "['*.png', '*.jpg', '*.jpeg', '*.webp', '*.ico', '*.mp3', '*.wav']"
    return f'''"""One-shot setup for {app}: gheasy schema, LFS, .env, the deploy workflow, secrets.

Generated by {generator()} from the lego pattern.
    python setup.py            # everything below, in order
    python setup.py mkenv      # write .env from the schema
    python setup.py workflow   # regenerate .github/workflows/deploy.yml
    python setup.py push       # push .env values to GitHub as secrets/variables
    python setup.py ssh-key    # upload ~/.ssh/<SERVER_NAME> as the DEPLOY_KEY secret
    python setup.py skills     # install the bundled SKILL.md files for agents
"""
import os, sys, importlib
from fastcore.all import Path, parse_env, filter_keys, in_
from dockeasy import env_set, env_get
from gheasy import GheasyConfig, gh_lfs, gh_push_env, gh_deploy_key_setup, repo_root
from gheasy.workflow import Workflow

__all__ = ['setup', 'mk_env', 'env2push', 'push_gh_vars', 'push_ssh_key', 'gen_deploy_workflow']

ROOT = repo_root()
APP = {app!r}
LFS_PATTERNS = {patterns}

# The schema, and the only place keys are declared. `None` -> GitHub secret; a string ->
# GitHub variable with that default. gheasy reads this to route each value; the workflow's
# `env:` block is this dict rendered; `.env` is this dict materialised.
ENV_KEYS = dict(
    MODE='prod', PORT={int(port)!r}, DOMAIN={host!r},
    SERVER_NAME=None, SERVER_USER='deploy', SERVER_PASSWORD=None,
    HCLOUD_TOKEN=None, CLOUDFLARE_API_TOKEN=None, CF_TUNNEL_TOKEN=None,
    RSYNC_FORCE='false',
)

def _load_env():
    envf = ROOT/'.env'
    return dict(os.environ) | (parse_env(fn=str(envf)) if envf.exists() else {{}})

def env2push():
    'Schema defaults overlaid with whatever is actually set locally.'
    return ENV_KEYS | filter_keys(_load_env(), in_(ENV_KEYS))

def init_gheasy():
    cfg = GheasyConfig.load(ROOT)
    cfg.app = cfg.app or APP
    cfg.env_schema = {{**ENV_KEYS, **(cfg.env_schema or {{}})}}
    cfg.save(ROOT)
    print(f'gheasy: {{len(cfg.env_schema)}} keys in the schema for {{cfg.app}}')

def lfs():
    gh_lfs(LFS_PATTERNS, path=str(ROOT))
    print(f'lfs: tracking {{len(LFS_PATTERNS)}} patterns')

def mk_env(env=None, path=None):
    'Write .env.example (or .env) from the schema, keeping any values already set.'
    env, path = env or ENV_KEYS, Path(path or ROOT/'.env.example')
    for k, v in env.items(): env_set(k, v if v is not None else '', path)
    print(f'env: wrote {{path}}')

def push_gh_vars(dry_run=False):
    'Push local values to GitHub. None-default keys become secrets; the rest variables.'
    to_push = env2push()
    gh_push_env(to_push, dry_run=dry_run, path=ROOT)
    print(f'push: {{"would push" if dry_run else "pushed"}} {{len(to_push)}} keys')

def push_ssh_key():
    'Upload ~/.ssh/<SERVER_NAME> as the DEPLOY_KEY secret, which is the key vpseasy deploys with.'
    name = env_get('SERVER_NAME', path=ROOT/'.env', default=APP)
    gh_deploy_key_setup(Path.home()/'.ssh'/name)

def gen_deploy_workflow():
    'The deploy workflow, with the schema rendered into its env block.'
    env = {{k: (f'${{{{{{{{ secrets.{{k}} }}}}}}}}' if v is None else f'${{{{{{{{ vars.{{k}} }}}}}}}}')
           for k, v in ENV_KEYS.items()}}
    env['DEPLOY_KEY'] = '${{{{ secrets.DEPLOY_KEY }}}}'
    ssh = ('mkdir -p ~/.ssh && name="${{SERVER_NAME:-' + APP + '}}" '
           '&& echo "$DEPLOY_KEY" > ~/.ssh/"$name" && chmod 600 ~/.ssh/"$name"')
    wf = Workflow('deploy')
    wf.on.push(branches=['main'])
    (wf.job('deploy').runs_on('ubuntu-latest').env(**env)
       .checkout().with_(lfs=True).end_step()
       .setup_uv().with_(python_version='3.13').end_step()
       .uv_install('uv sync --group dev').end_step()
       .step('Install SSH key').if_("env.DEPLOY_KEY != ''").run(ssh).end_step()
       .step('Deploy').run('python deploy.py deploy').end_job())
    p = ROOT/'.github'/'workflows'/'deploy.yml'
    wf.build().save(p)
    print(f'workflow: wrote {{p}}')

def install_skills():
    'Copy each installed package SKILL.md into .agents/ and .claude/ so agents can read them.'
    for nm in ('dockeasy', 'gheasy', 'vpseasy', 'cfeasy'):
        try: mod = importlib.import_module(nm)
        except ImportError: print(f'skip {{nm}}: not installed'); continue
        mv = getattr(mod, 'mv_skill_md', None)
        if mv: mv(dry_run=False)

def setup():
    init_gheasy()
    lfs()
    mk_env()
    gen_deploy_workflow()
    install_skills()
    print('Setup complete. Fill in .env, then `python setup.py push` and `python setup.py ssh-key`.')

if __name__ == '__main__':
    arg = (sys.argv[1:] or [''])[0]
    if arg == 'push': push_gh_vars('--dry-run' in sys.argv)
    elif arg == 'mkenv': mk_env(env2push(), path=ROOT/'.env')
    elif arg == 'workflow': gen_deploy_workflow()
    elif arg == 'ssh-key': push_ssh_key()
    elif arg == 'skills': install_skills()
    else: setup()
'''

`setup_template` answers with the text of a `setup.py`. It writes nothing.

The generated file declares `ENV_KEYS` once and every other part of it reads that dict. `.env` is the dict materialised, the push routes each key to a secret or a variable by whether its default is `None`, and the workflow's `env:` block is the dict rendered.

`subdomain` and `domain` compose the host. `app.example.com` comes from `subdomain='app'` with `domain='example.com'`, and a bare `domain` is the host on its own.

In [ ]:
#| hide
def literal(src, name):
    "The value of one top-level assignment in generated source, without importing the file."
    node = first(n for n in ast.parse(src).body
                 if isinstance(n, ast.Assign) and getattr(n.targets[0], 'id', '') == name)
    return eval(compile(ast.Expression(node.value), '<generated>', 'eval'))

In [ ]:
src = setup_template('lego', 'example.com', subdomain='app')
literal(src, 'ENV_KEYS')

{'MODE': 'prod',
 'PORT': 5001,
 'DOMAIN': 'app.example.com',
 'SERVER_NAME': None,
 'SERVER_USER': 'deploy',
 'SERVER_PASSWORD': None,
 'HCLOUD_TOKEN': None,
 'CLOUDFLARE_API_TOKEN': None,
 'CF_TUNNEL_TOKEN': None,
 'RSYNC_FORCE': 'false'}

In [ ]:
#| hide
compile(src, 'setup.py', 'exec')   # the brace escaping in the template is easy to get wrong
keys = literal(src, 'ENV_KEYS')
assert set(DEPLOY_KEYS) <= set(keys), 'the pipeline names a key the generated setup never pushes'
test_eq(keys['DOMAIN'], 'app.example.com')
test_eq(literal(setup_template('lego', 'example.com'), 'ENV_KEYS')['DOMAIN'], 'example.com')

In [ ]:
#| export
def deploy_template(app, domain, subdomain='', port=5001, second_host=''):
    "lego's `deploy.py` in full, with this project's names in it. `second_host` is lego's apex."
    host = f'{subdomain}.{domain}' if subdomain else domain
    tunnel = f'{subdomain or app}_{domain}'
    second = f'''

# The second hostname: same server, same container, same tunnel. Caddy tells them apart by
# the Host header, and `or` rather than a getenv default because the workflow passes every
# key as `${{ vars.KEY }}`. An unset repository variable arrives as the empty string, not as
# absent, so a getenv default would not fire.
second_host = os.getenv('SECOND_HOST') or {second_host!r}''' if second_host else (
    '\n\nsecond_host = os.getenv("SECOND_HOST") or ""')
    return f'''"""Deploy {app} to a VPS: dockeasy image, vpseasy stack and server, cfeasy tunnel.

Generated by {generator()} from the lego pattern. Edit freely -- it is your deploy, and the panel
runs whatever this file says.
    python deploy.py env       # write .env from the environment store
    python deploy.py compose   # Dockerfile + docker-compose.yml + Caddyfile
    python deploy.py deploy    # tunnel, DNS, server, rsync, up
    python deploy.py status    # what is deployed where
    python deploy.py nuke      # delete the server and the tunnel
"""
import os, sys, secrets
from fastcore.all import Path
from dockeasy import detect_app, env_set, env_get
from cfeasy import CF
from vpseasy import caddy_stack, hetzner_deploy, Hetzner

root = Path(__file__).resolve().parent
app, domain, subdomain = {app!r}, {domain!r}, {subdomain!r}
host = {host!r}
tunnel_name = {tunnel!r}
srv, app_svc, port = '/srv/app', 'app', {int(port)}{second}

# Extra apt packages the image needs, and the paths that must survive a redeploy. `vols`
# become bind mounts under the deploy directory, so data outlives the container.
pkgs = ['curl']
vols = ['/app/data', '/app/backups']

# What rsync uploads, and what it must not. The include list is explicit because a deploy
# that accidentally ships .venv or .git is slow the first time and wrong every time after.
inc = ['{app}/', 'static/', 'pyproject.toml', 'uv.lock', 'main.py', 'Dockerfile',
       'docker-compose.yml', 'Caddyfile', '.env']
exc = ['data/', 'backups/', '.git/', '.venv/', '__pycache__/']

RSYNC_FORCE = {{'checksum': '--checksum', 'ignore-times': '--ignore-times'}}

def caddy_site(name, *directives):
    \'\'\'One site block for the shared Caddy.
    `http://` because the tunnel terminates TLS in front of it -- there is no public port 80
    to run an ACME challenge against.\'\'\'
    body = ''.join(f'\\t{{d}}\\n' for d in directives)
    return f'http://{{name}} {{{{\\n{{body}}\\treverse_proxy {{app_svc}}:{{port}}\\n}}}}\\n'

def mk_caddyfile(path=None):
    "Every hostname, one Caddy, one app container."
    path = Path(path or root/'Caddyfile')
    text = caddy_site(host)
    if second_host: text += caddy_site(second_host)
    path.write_text(text)
    print(f'caddy: {{host}}{{" + " + second_host if second_host else ""}} -> {{app_svc}}:{{port}}')

def mk_compose():
    "dockeasy detects what this project builds; vpseasy wraps it in Caddy and cloudflared."
    df = detect_app(str(root), pkgs=pkgs, vols=vols)
    c = caddy_stack(host, df, vols=vols, root=root)
    mk_caddyfile()
    return c

def mk_env():
    "Materialise the values the container reads. The deploy uploads this file."
    for k, v in (('MODE', 'prod'), ('PORT', str(port)), ('DOMAIN', host)):
        env_set(k, env_get(k, default=v), path=root/'.env')
    print(f'env: wrote {{root/".env"}}')

def add_second_dns(cf, tid):
    \'\'\'Point the extra hostname at the tunnel that already exists.
    One tunnel, not two: cloudflared runs against the same Caddy, so every hostname routed
    through it arrives in the same place. A failure here costs that hostname and nothing
    else, so it warns with the record to add by hand rather than aborting a deploy that is
    otherwise fine.\'\'\'
    if not second_host: return
    try:
        cf.tunnel_cname(second_host, second_host, tid)
        print(f'dns: {{second_host}} -> tunnel {{tid}}')
    except Exception as e:
        print(f'WARNING: could not point {{second_host}} at the tunnel: {{e}}\\n'
              f'         {{host}} is unaffected. Add a proxied CNAME '
              f'{{second_host}} -> {{tid}}.cfargotunnel.com by hand.')

def deploy(force=None):
    \'\'\'Idempotent: the tunnel is reused if it exists, the server provisioned only if it does not.
    force= '' | 'checksum' | 'ignore-times' -- rsync skips by size and mtime, which is wrong
    exactly when a build wrote the same bytes at a different time.\'\'\'
    mk_env()
    mk_compose()
    cf = CF(token=env_get('CLOUDFLARE_API_TOKEN', path=root/'.env'))
    tid, tok = cf.setup_tunnel(domain, subdomain or None, tunnel_name=tunnel_name)
    print('cloudflare tunnel:', tid)
    add_second_dns(cf, tid)
    env_set('CF_TUNNEL_TOKEN', tok, path=root/'.env')
    extra = RSYNC_FORCE.get(force or os.getenv('RSYNC_FORCE', ''))
    if extra: print(f'rsync force: {{extra}}')
    name = env_get('SERVER_NAME', path=root/'.env', default=app)
    user = env_get('SERVER_USER', path=root/'.env', default='deploy')
    pw = env_get('SERVER_PASSWORD', path=root/'.env', default=None)
    key = env_get('HETZNER_KEY', path=root/'.env', default=None)
    r = hetzner_deploy(name, root, include=inc, exclude=exc, path=srv, extra=extra,
                       password=pw, user=user, key=key)
    env_set('HETZNER_IP', r.ip, path=root/'.env')
    if r.key: env_set('HETZNER_KEY', r.key, path=root/'.env')
    print(f'deployed: https://{{host}} at {{r.ip}}')

def status():
    "Which servers exist on this token, and which tunnels."
    name = env_get('SERVER_NAME', path=root/'.env', default=app)
    for s in Hetzner().servers(): print(f'server {{s["name"]:20}} {{s["ip"]:16}} {{s["status"]}}')
    cf = CF(token=env_get('CLOUDFLARE_API_TOKEN', path=root/'.env'))
    for t in cf.tunnels(): print(f'tunnel {{t.get("name","")!r:22}} {{t.get("id","")}}')
    print(f'this project deploys as {{name}} -> {{host}}')

def nuke():
    "Delete the server and the tunnel. Irreversible, so it asks for a token you have to read."
    typ = secrets.token_urlsafe(8)
    if input(f'This deletes the server and the tunnel. Type {{typ}} to proceed: ') != typ:
        return print('aborted')
    name = env_get('SERVER_NAME', path=root/'.env', default=app)
    Hetzner().delete(name)
    print(f'server {{name}} deleted')
    try:
        cf = CF(token=env_get('CLOUDFLARE_API_TOKEN', path=root/'.env'))
        cf.delete_tunnel(cf.tunnel_id(tunnel_name))
        print('tunnel deleted')
    except Exception as e: print(f'tunnel not deleted: {{e}}')

if __name__ == '__main__':
    args = sys.argv[1:]
    cmd = (args or [''])[0]
    if cmd == 'compose': mk_compose()
    elif cmd == 'env': mk_env()
    elif cmd == 'deploy': deploy(force=args[1] if len(args) > 1 else None)
    elif cmd == 'status': status()
    elif cmd == 'nuke': nuke()
    else: print('usage: deploy.py env | compose | deploy | status | nuke')
'''

`deploy_template` answers with the text of a `deploy.py`. It writes nothing.

The generated script is idempotent. The tunnel is reused where one exists, and the server is provisioned only where there is none, so running it twice deploys twice and creates once.

`second_host` is a second hostname on the same server, the same container and the same tunnel. Caddy tells the two apart by the `Host` header. Where the DNS record for the second cannot be written, the script prints the record to add by hand and carries on. That failure costs one hostname, and the first is already served.

The generated script reads `SECOND_HOST` with `or` rather than a `getenv` default. The workflow passes every key as a `vars` expression, and an unset repository variable arrives as the empty string rather than as absent, so a default would not fire.

In [ ]:
dsrc = deploy_template('lego', 'example.com', subdomain='app')
literal(dsrc, 'host'), literal(dsrc, 'tunnel_name'), literal(dsrc, 'inc')[:4]

('app.example.com',
 'app_example.com',
 ['lego/', 'static/', 'pyproject.toml', 'uv.lock'])

In [ ]:
#| hide
for text in (dsrc, deploy_template('lego', 'example.com', second_host='example.org')):
    compile(text, 'deploy.py', 'exec')
test_eq(literal(deploy_template('lego', 'example.com'), 'host'), 'example.com')
test_eq(literal(deploy_template('lego', 'example.com'), 'tunnel_name'), 'lego_example.com')
assert not set(literal(dsrc, 'inc')) & set(literal(dsrc, 'exc')), 'nothing is both sent and excluded'
for s in DEPLOY_STEPS:
    sub = s.cmd.split()[-1]
    assert f'cmd == {sub!r}' in dsrc, f'{sub} is a step, but the generated script has no such subcommand'

In [ ]:
#| export
class Deploy(Pipeline):
    "The deploy pipeline: the project's own `deploy.py`, one subcommand per step."
    FILE = DEPLOY_FILE
    KIND = 'deploy'
    @staticmethod
    def defaults(root): return list(DEPLOY_STEPS)
    def target(self):
        "Where this deploy goes and what it is still missing, from the store the steps will read."
        # Whatever answers `get`, not only an `EnvStore`: a host keeps its own store, and the
        # class check sent every value read here to a second one that had never been configured.
        store = self.env if hasattr(self.env, 'get') else EnvStore()
        def value(key, default=''): return env_value(store, key, default)
        missing = [k for k, v in DEPLOY_KEYS.items()
                   if v is None and not env_value(store, k, secret=True)]
        return {'app': value('APP') or self.root.name, 'domain': value('DOMAIN'),
            'server': value('SERVER_NAME'), 'user': value('SERVER_USER', 'deploy'),
            'port': value('PORT', '5001'), 'missing': missing,
            'script': (self.root/'deploy.py').exists(),
            'setup': (self.root/'setup.py').exists(),
            'compose': (self.root/'docker-compose.yml').exists(),
            'dockerfile': (self.root/'Dockerfile').exists(),
            'caddyfile': (self.root/'Caddyfile').exists(),
            'keys': {k: (v is None) for k, v in DEPLOY_KEYS.items()}}
    def state(self): return super().state() | {'target': self.target()}
    def scaffold(self, domain='', subdomain='', app='', force=False, second_host=''):
        "Write lego's pattern out: the one-shot `setup.py` and the `deploy.py` CI runs on every push."
        domain = str(domain or '').strip()
        if not domain: raise DeployError('a domain is required — the host this app answers on')
        app = str(app or '').strip() or self.root.name
        subdomain, second_host = str(subdomain or '').strip(), str(second_host or '').strip()
        script = self.root/'deploy.py'
        if script.exists() and not force: raise DeployError('this project already has a deploy.py')
        port = int(self.target()['port'] or 5001)
        written = []
        script.write_text(deploy_template(app, domain, subdomain, port, second_host), encoding='utf-8')
        written.append(str(script))
        setup = self.root/'setup.py'
        if not setup.exists() or force:
            setup.write_text(setup_template(app, domain, subdomain, port), encoding='utf-8')
            written.append(str(setup))
        try:
            from gheasy.core import GheasyConfig, cfg_path
            cfg = GheasyConfig.load(str(self.root))
            cfg.app = cfg.app or app
            cfg.env_schema = {**DEPLOY_KEYS, **(cfg.env_schema or {})}
            cfg.save(str(self.root))
            written.append(str(cfg_path(str(self.root))))
        except ImportError:
            pass
        try:
            from pullup.workflows import Workflows
            written.append(Workflows(self.root).add('deploy')['path'])
        except Exception:
            pass
        return {'written': written, 'state': self.state()}

`Deploy` is `Pipeline` with `DEPLOY_STEPS` as its default plan and `deploy.json` as its file.

`scaffold` writes the pattern out: `deploy.py`, a `setup.py` where there is none, the project's gheasy config with `DEPLOY_KEYS` merged into its schema, and `.github/workflows/deploy.yml`. It answers with the paths it wrote and the new state. The config and the workflow are written where gheasy is installed. Without gheasy only the two scripts are written, and `written` lists what it wrote.

A domain is required. `scaffold` refuses to overwrite an existing `deploy.py` unless `force`, so re-running it does not discard a deploy someone has edited.

In [ ]:
#| hide
tmp = TemporaryDirectory(); root = Path(tmp.name)/'lego'; root.mkdir()
(root/'pyproject.toml').write_text('[project]\nname = "lego"\n')

24

In [ ]:
d = Deploy(root)
[str(Path(p).relative_to(root)) for p in d.scaffold(domain='example.com', subdomain='app')['written']]

['deploy.py',
 'setup.py',
 '.gheasy/config.json',
 '.github/workflows/deploy.yml']

`target` says where this deploy goes and what it is still missing, read from the store the steps themselves will read. It reports which of the generated files exist, so a caller can tell a scaffolded project from one that has never been deployed.

In [ ]:
{k: v for k, v in d.target().items() if k != 'keys'}

{'app': 'lego',
 'domain': 'app.example.com',
 'server': '',
 'user': 'deploy',
 'port': '5001',
 'missing': ['SERVER_NAME',
  'HCLOUD_TOKEN',
  'CLOUDFLARE_API_TOKEN',
  'CF_TUNNEL_TOKEN'],
 'script': True,
 'setup': True,
 'compose': False,
 'dockerfile': False,
 'caddyfile': False}

In [ ]:
#| hide
t = d.target()
test_eq([t['app'], t['script'], t['setup'], t['compose'], t['dockerfile']], ['lego', True, True, False, False])
assert set(t['missing']) <= {k for k, v in DEPLOY_KEYS.items() if v is None}, 'a key with a default is never missing'
test_eq((t['keys']['DOMAIN'], t['keys']['SERVER_USER']), (True, False))
test_eq((d.state()['kind'], d.path.name, [s.id for s in d.steps]), ('deploy', 'deploy.json', ['env', 'compose', 'deploy']))
test_fail(lambda: d.scaffold(domain='example.com'), contains='already has a deploy.py')
test_fail(lambda: d.scaffold(domain=' '), contains='a domain is required')
wf = root/'.github'/'workflows'/'deploy.yml'
if wf.exists():
    yml = wf.read_text()
    for k in DEPLOY_KEYS: assert k in yml, f'{k} is in the schema but not in the deploy workflow'
tmp.cleanup()